# Binary Logit Robustness: `effect_proc_improve_high`

이 노트북은 `04_make_2024_featured.py`에서 생성한 이항화 종속변수 `effect_proc_improve_high`를 사용해 binary logistic regression 강건성 분석을 수행한다. 분석 위치는 주모형이 아니라 **DV robustness check / ceiling effect 대응 분석**이다.

기존 08/09번 노트북의 구조를 기준으로 두 가지 specification을 함께 확인한다.

- **H2 headline spec 복제:** `effect_proc_improve_high ~ ai_use_sum + it_org_any + it_invest_sum + ai_use_sum:it_org_any + controls`
- **Integrated binary logit spec:** 위 구조에 `dmi`와 `ai_use_sum:dmi`를 추가해 요청된 세 핵심항(`ai_use_sum`, `ai_use_sum:it_org_any`, `ai_use_sum:dmi`)을 함께 해석한다.


## 1. 설정 및 라이브러리

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from statsmodels.tools.sm_exceptions import ConvergenceWarning, PerfectSeparationError

try:
    from statsmodels.tools.sm_exceptions import PerfectSeparationWarning
except ImportError:
    PerfectSeparationWarning = Warning

try:
    from IPython.display import display, Markdown
except ImportError:
    class Markdown(str):
        pass

    def display(obj):
        print(obj)

SAVE_OUTPUTS = True

BASE_DIR = Path.cwd().resolve()
if BASE_DIR.name == "code":
    BASE_DIR = BASE_DIR.parent

DATA_PATH = BASE_DIR / "working" / "featured" / "nia_2024_featured.csv"
OUTPUT_DIR = BASE_DIR / "outputs" / "09_alternative_robustness_models"
TABLE_DIR = OUTPUT_DIR / "tables"
MODEL_DIR = OUTPUT_DIR / "models"

if SAVE_OUTPUTS:
    TABLE_DIR.mkdir(parents=True, exist_ok=True)
    MODEL_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 180)
pd.set_option("display.width", 220)

print("BASE_DIR:", BASE_DIR)
print("DATA_PATH:", DATA_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)


## 2. 데이터 로드 및 변수 정의

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f"featured 데이터셋을 찾지 못했습니다: {DATA_PATH}")

df = pd.read_csv(DATA_PATH, low_memory=False)
LOADED_SHAPE = df.shape

DV_BINARY = "effect_proc_improve_high"
DV_ORIGINAL = "effect_proc_improve"
AI = "ai_use_sum"
IT_ORG = "it_org_any"
DMI = "dmi"
INVEST_CONTROL = "it_invest_sum"
BASE_CONTROLS = ["firm_size", "industry", "region", "firm_type"]
YEAR = "year"

# 기존 08/09번 노트북은 firm_size, industry, region, firm_type을 통제한다.
# year는 현재 featured 데이터에 존재하지만 2024 단일연도이면 FE로 추가하지 않는다.
year_unique_n = df[YEAR].nunique(dropna=True) if YEAR in df.columns else 0
YEAR_CONTROLS = [YEAR] if year_unique_n > 1 else []
CONTROLS = BASE_CONTROLS + YEAR_CONTROLS

print("사용 데이터 파일:", DATA_PATH.relative_to(BASE_DIR))
print("데이터 shape:", LOADED_SHAPE)
print("year unique N:", year_unique_n)
print("사용 통제변수:", CONTROLS)


## 3. 데이터 점검

In [ ]:
REQUIRED_VARS = [
    DV_BINARY,
    DV_ORIGINAL,
    AI,
    IT_ORG,
    DMI,
    INVEST_CONTROL,
    "firm_size",
    "industry",
    "region",
    "firm_type",
] + YEAR_CONTROLS

variable_check = pd.DataFrame([
    {
        "variable": var,
        "exists": var in df.columns,
        "valid_N": int(df[var].notna().sum()) if var in df.columns else np.nan,
        "missing_N": int(df[var].isna().sum()) if var in df.columns else np.nan,
        "dtype": str(df[var].dtype) if var in df.columns else "missing",
    }
    for var in REQUIRED_VARS
])
display(variable_check)

missing_vars = variable_check.loc[~variable_check["exists"], "variable"].tolist()
if missing_vars:
    raise ValueError("Binary logit 분석에 필요한 변수가 없습니다: " + ", ".join(missing_vars))

analysis_df = df[REQUIRED_VARS].copy()
for var in REQUIRED_VARS:
    analysis_df[var] = pd.to_numeric(analysis_df[var], errors="coerce")

print("effect_proc_improve_high value counts:")
display(analysis_df[DV_BINARY].value_counts(dropna=False).sort_index().rename_axis(DV_BINARY).reset_index(name="N"))

print("effect_proc_improve_high 결측치 개수:", int(analysis_df[DV_BINARY].isna().sum()))

print("effect_proc_improve x effect_proc_improve_high crosstab:")
display(pd.crosstab(
    analysis_df[DV_ORIGINAL].astype("Int64").astype("string").fillna("<NA>"),
    analysis_df[DV_BINARY].astype("Int64").astype("string").fillna("<NA>"),
    rownames=[DV_ORIGINAL],
    colnames=[DV_BINARY],
    dropna=False,
))

missing_summary = pd.DataFrame({
    "variable": REQUIRED_VARS,
    "valid_N": [int(analysis_df[var].notna().sum()) for var in REQUIRED_VARS],
    "missing_N": [int(analysis_df[var].isna().sum()) for var in REQUIRED_VARS],
})
display(missing_summary)


## 4. Formula 구성

In [ ]:
def control_formula(controls: list[str]) -> str:
    terms = []
    for var in controls:
        if var == "firm_size":
            terms.append("firm_size")
        elif var in ["industry", "region", "firm_type", "year"]:
            terms.append(f"C({var})")
        else:
            terms.append(var)
    return " + ".join(terms)

CONTROL_TERMS = control_formula(CONTROLS)
CONTROL_SUFFIX = f" + {CONTROL_TERMS}" if CONTROL_TERMS else ""

H2_HEADLINE_FORMULA = (
    f"{DV_BINARY} ~ {AI} + {IT_ORG} + {INVEST_CONTROL} + "
    f"{AI}:{IT_ORG}{CONTROL_SUFFIX}"
)

INTEGRATED_BINARY_LOGIT_FORMULA = (
    f"{DV_BINARY} ~ {AI} + {IT_ORG} + {DMI} + {INVEST_CONTROL} + "
    f"{AI}:{IT_ORG} + {AI}:{DMI}{CONTROL_SUFFIX}"
)

MODEL_SPECS = {
    "H2 Headline Binary Logit": {
        "formula": H2_HEADLINE_FORMULA,
        "required": [DV_BINARY, AI, IT_ORG, INVEST_CONTROL] + CONTROLS,
        "role": "Exact H2 headline Model 3 spec from 08_main_model_results with binary DV",
    },
    "Integrated Binary Logit": {
        "formula": INTEGRATED_BINARY_LOGIT_FORMULA,
        "required": [DV_BINARY, AI, IT_ORG, DMI, INVEST_CONTROL] + CONTROLS,
        "role": "Integrated DV robustness spec including H2 and DMI interaction terms",
    },
}

formula_table = pd.DataFrame([
    {"model_name": name, "role": spec["role"], "formula": spec["formula"]}
    for name, spec in MODEL_SPECS.items()
])
display(formula_table)


## 5. Logistic Regression 실행 및 진단

In [ ]:
def p_stars(p_value: float) -> str:
    if pd.isna(p_value):
        return ""
    if p_value < 0.001:
        return "***"
    if p_value < 0.01:
        return "**"
    if p_value < 0.05:
        return "*"
    if p_value < 0.1:
        return "†"
    return ""


def direction(coef: float) -> str:
    if pd.isna(coef):
        return ""
    if coef > 0:
        return "+"
    if coef < 0:
        return "-"
    return "0"


def diagnose_binary_cells(model_data: pd.DataFrame, predictors: list[str]) -> pd.DataFrame:
    rows = []
    for var in predictors:
        if var == DV_BINARY or var not in model_data.columns:
            continue
        nunique = model_data[var].nunique(dropna=True)
        if nunique <= 30:
            tab = pd.crosstab(model_data[var], model_data[DV_BINARY], dropna=False)
            has_zero_cell = bool((tab == 0).any().any()) if not tab.empty else False
            rows.append({
                "variable": var,
                "n_unique": int(nunique),
                "has_zero_outcome_cell": has_zero_cell,
                "min_cell_count": int(tab.min().min()) if not tab.empty else np.nan,
            })
        else:
            grouped = pd.qcut(model_data[var], q=10, duplicates="drop")
            tab = pd.crosstab(grouped, model_data[DV_BINARY], dropna=False)
            rows.append({
                "variable": var,
                "n_unique": int(nunique),
                "has_zero_outcome_cell": bool((tab == 0).any().any()) if not tab.empty else False,
                "min_cell_count": int(tab.min().min()) if not tab.empty else np.nan,
            })
    return pd.DataFrame(rows)


def fit_logit_hc3(model_name: str, spec: dict):
    model_vars = list(dict.fromkeys(spec["required"]))
    model_data = analysis_df[model_vars].dropna().copy()
    model_data = model_data[model_data[DV_BINARY].isin([0, 1])].copy()

    y_counts = model_data[DV_BINARY].value_counts().to_dict()
    if len(y_counts) < 2:
        raise ValueError(f"{model_name}: 종속변수가 한 범주만 남아 logit을 추정할 수 없습니다. counts={y_counts}")

    warning_messages = []
    try:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            result = smf.logit(spec["formula"], data=model_data).fit(
                disp=False,
                maxiter=200,
                cov_type="HC3",
            )
            warning_messages = [str(w.message) for w in caught]
    except PerfectSeparationError as exc:
        diagnostics = diagnose_binary_cells(model_data, model_vars)
        return None, model_data, "perfect separation: " + str(exc), warning_messages, diagnostics
    except Exception as exc:
        diagnostics = diagnose_binary_cells(model_data, model_vars)
        return None, model_data, f"failed: {type(exc).__name__}: {exc}", warning_messages, diagnostics

    converged = bool(result.mle_retvals.get("converged", False)) if hasattr(result, "mle_retvals") else True
    status = "ok" if converged else "not converged"
    diagnostics = diagnose_binary_cells(model_data, model_vars)
    return result, model_data, status, warning_messages, diagnostics


LOGIT_RESULTS = {}
LOGIT_MODEL_DATA = {}
FIT_ROWS = []
DIAGNOSTICS = {}
FULL_SUMMARIES = []

for model_name, spec in MODEL_SPECS.items():
    result, model_data, status, warning_messages, diagnostics = fit_logit_hc3(model_name, spec)
    LOGIT_RESULTS[model_name] = result
    LOGIT_MODEL_DATA[model_name] = model_data
    DIAGNOSTICS[model_name] = diagnostics
    FIT_ROWS.append({
        "model_name": model_name,
        "estimator": "statsmodels Logit, HC3 robust SE",
        "N": int(len(model_data)),
        "status": status,
        "converged": bool(result.mle_retvals.get("converged", False)) if result is not None and hasattr(result, "mle_retvals") else False,
        "pseudo_R2_McFadden": float(result.prsquared) if result is not None else np.nan,
        "log_likelihood": float(result.llf) if result is not None else np.nan,
        "AIC": float(result.aic) if result is not None else np.nan,
        "BIC": float(result.bic) if result is not None else np.nan,
        "warnings": " | ".join(warning_messages),
        "formula": spec["formula"],
    })
    print(f"{model_name}: status={status}, N={len(model_data):,}")
    if warning_messages:
        print("Warnings:", " | ".join(warning_messages))
    display(diagnostics)
    if result is not None:
        FULL_SUMMARIES.append(f"{'=' * 100}\n{model_name}\n{spec['formula']}\n{'=' * 100}\n{result.summary()}")

fit_stats = pd.DataFrame(FIT_ROWS)
for col in ["pseudo_R2_McFadden", "log_likelihood", "AIC", "BIC"]:
    fit_stats[col] = pd.to_numeric(fit_stats[col], errors="coerce").round(4)
display(fit_stats)


## 6. 결과표

In [ ]:
def result_table_for_model(model_name: str, result) -> pd.DataFrame:
    if result is None:
        return pd.DataFrame()
    conf = result.conf_int()
    table = pd.DataFrame({
        "model_name": model_name,
        "term": result.params.index,
        "coefficient": result.params.values,
        "standard_error": result.bse.values,
        "z_value": result.tvalues.values,
        "p_value": result.pvalues.values,
        "ci_lower": conf.iloc[:, 0].values,
        "ci_upper": conf.iloc[:, 1].values,
    })
    table["odds_ratio"] = np.exp(table["coefficient"])
    table["odds_ratio_ci_lower"] = np.exp(table["ci_lower"])
    table["odds_ratio_ci_upper"] = np.exp(table["ci_upper"])
    table["stars"] = table["p_value"].map(p_stars)
    table["direction"] = table["coefficient"].map(direction)
    table["N"] = int(result.nobs)
    table["pseudo_R2_McFadden"] = float(result.prsquared)
    table["log_likelihood"] = float(result.llf)
    return table

binary_logit_results = pd.concat(
    [result_table_for_model(name, result) for name, result in LOGIT_RESULTS.items() if result is not None],
    ignore_index=True,
)

binary_logit_results_display = binary_logit_results.copy()
for col in [
    "coefficient", "standard_error", "z_value", "p_value", "ci_lower", "ci_upper",
    "odds_ratio", "odds_ratio_ci_lower", "odds_ratio_ci_upper", "pseudo_R2_McFadden", "log_likelihood",
]:
    binary_logit_results_display[col] = pd.to_numeric(binary_logit_results_display[col], errors="coerce").round(4)

display(binary_logit_results_display)

CORE_TERMS = [AI, f"{AI}:{IT_ORG}", f"{AI}:{DMI}"]
core_binary_logit_results = binary_logit_results[binary_logit_results["term"].isin(CORE_TERMS)].copy()
core_display = core_binary_logit_results.copy()
for col in [
    "coefficient", "standard_error", "z_value", "p_value", "ci_lower", "ci_upper",
    "odds_ratio", "odds_ratio_ci_lower", "odds_ratio_ci_upper", "pseudo_R2_McFadden", "log_likelihood",
]:
    core_display[col] = pd.to_numeric(core_display[col], errors="coerce").round(4)
display(core_display)


## 7. 핵심 계수 해석 및 OLS 방향 비교

In [ ]:
OLS_REFERENCE = {
    AI: {"direction": "-", "pattern": "기존 integrated OLS에서 AI 활용 범위 단독항은 음(-)"},
    f"{AI}:{IT_ORG}": {"direction": "+", "pattern": "기존 Model 3/integrated OLS에서 AI 활용 범위 × IT 조직 보유 항은 양(+)"},
    f"{AI}:{DMI}": {"direction": "-", "pattern": "기존 integrated OLS에서 AI 활용 범위 × DMI 항은 약하거나 불안정"},
}


def sig_text(p_value: float) -> str:
    if pd.isna(p_value):
        return "확인 불가"
    if p_value < 0.05:
        return "통계적으로 유의함"
    if p_value < 0.1:
        return "10% 수준에서 약하게 유의함"
    return "통계적으로 뚜렷하지 않음"


def term_sentence(row: pd.Series) -> str:
    term = row["term"]
    coef = row["coefficient"]
    p_value = row["p_value"]
    or_value = row["odds_ratio"]
    direction_text = "양(+)" if coef > 0 else "음(-)" if coef < 0 else "0"
    return (
        f"`{term}`은 {direction_text} 방향의 계수({coef:.3f}{p_stars(p_value)}, "
        f"p={p_value:.4f})로 나타났고, odds ratio는 {or_value:.3f}이다. "
        f"유의성은 {sig_text(p_value)}."
    )

primary_model_name = "Integrated Binary Logit"
primary_core = core_binary_logit_results[core_binary_logit_results["model_name"].eq(primary_model_name)].copy()

comparison_rows = []
for _, row in primary_core.iterrows():
    term = row["term"]
    ref = OLS_REFERENCE.get(term, {})
    logit_direction = direction(row["coefficient"])
    expected_direction = ref.get("direction")
    if term == f"{AI}:{DMI}":
        pattern_status = "약하거나 불안정한 패턴 유지" if row["p_value"] >= 0.05 or p_stars(row["p_value"]) == "†" else "logit에서는 더 뚜렷함"
    else:
        pattern_status = "방향성 유지" if logit_direction == expected_direction else "방향성 불일치"
    comparison_rows.append({
        "term": term,
        "OLS_reference_pattern": ref.get("pattern"),
        "OLS_expected_direction": expected_direction,
        "logit_direction": logit_direction,
        "logit_p_value": row["p_value"],
        "logit_significance": sig_text(row["p_value"]),
        "comparison": pattern_status,
    })

binary_logit_ols_direction_comparison = pd.DataFrame(comparison_rows)
display(binary_logit_ols_direction_comparison)

interpretation_lines = [term_sentence(row) for _, row in primary_core.iterrows()]
interpretation_lines.append(
    "종합하면 binary logit에서도 `ai_use_sum` 단독항은 음(-), `ai_use_sum × it_org_any`는 양(+)으로 나타나 기존 OLS의 핵심 방향성이 유지된다. "
    "반면 `ai_use_sum × dmi`는 계수 크기와 유의성이 약해, DMI 상호작용은 기존 OLS와 마찬가지로 안정적인 headline 패턴으로 보기 어렵다."
)

display(Markdown("**핵심 계수 해석**\n" + "\n".join(f"- {line}" for line in interpretation_lines)))


## 8. 결과 저장

In [ ]:
if SAVE_OUTPUTS:
    TABLE_DIR.mkdir(parents=True, exist_ok=True)
    MODEL_DIR.mkdir(parents=True, exist_ok=True)

    result_csv = TABLE_DIR / "binary_logit_effect_proc_improve_high_results.csv"
    result_xlsx = TABLE_DIR / "binary_logit_effect_proc_improve_high_results.xlsx"
    fit_csv = TABLE_DIR / "binary_logit_effect_proc_improve_high_fit_stats.csv"
    comparison_csv = TABLE_DIR / "binary_logit_effect_proc_improve_high_ols_comparison.csv"
    full_summary_path = MODEL_DIR / "binary_logit_effect_proc_improve_high_full_summary.txt"

    binary_logit_results.to_csv(result_csv, index=False, encoding="utf-8-sig")
    fit_stats.to_csv(fit_csv, index=False, encoding="utf-8-sig")
    binary_logit_ols_direction_comparison.to_csv(comparison_csv, index=False, encoding="utf-8-sig")

    with pd.ExcelWriter(result_xlsx) as writer:
        binary_logit_results.to_excel(writer, sheet_name="all_coefficients", index=False)
        core_binary_logit_results.to_excel(writer, sheet_name="core_terms", index=False)
        fit_stats.to_excel(writer, sheet_name="fit_stats", index=False)
        formula_table.to_excel(writer, sheet_name="formulas", index=False)
        variable_check.to_excel(writer, sheet_name="variable_check", index=False)
        missing_summary.to_excel(writer, sheet_name="missing_summary", index=False)
        binary_logit_ols_direction_comparison.to_excel(writer, sheet_name="ols_comparison", index=False)

    full_summary_path.write_text("\n\n".join(FULL_SUMMARIES), encoding="utf-8")

    print("저장 완료:")
    print("-", result_csv.relative_to(BASE_DIR))
    print("-", result_xlsx.relative_to(BASE_DIR))
    print("-", fit_csv.relative_to(BASE_DIR))
    print("-", comparison_csv.relative_to(BASE_DIR))
    print("-", full_summary_path.relative_to(BASE_DIR))
else:
    print("SAVE_OUTPUTS=False이므로 저장하지 않았습니다.")


## 9. 4.6절 본문용 자동 해석 문단

In [ ]:
def get_primary(term: str) -> pd.Series:
    row = primary_core[primary_core["term"].eq(term)]
    if row.empty:
        raise ValueError(f"핵심항 결과가 없습니다: {term}")
    return row.iloc[0]

ai_row = get_primary(AI)
h2_row = get_primary(f"{AI}:{IT_ORG}")
h3_row = get_primary(f"{AI}:{DMI}")

paper_paragraph = (
    "Ceiling effect 가능성에 대응하기 위해 프로세스 개선 효과 인식을 1~3점과 4~5점으로 이항화한 "
    "`effect_proc_improve_high`를 종속변수로 두고 binary logistic regression을 추가 추정하였다. "
    f"통합 specification은 `{INTEGRATED_BINARY_LOGIT_FORMULA}`이며, complete-case 기준 N은 {int(LOGIT_RESULTS[primary_model_name].nobs):,}이다. "
    f"분석 결과 `ai_use_sum` 단독항은 {direction(ai_row['coefficient'])} 방향(coef={ai_row['coefficient']:.3f}{p_stars(ai_row['p_value'])}, "
    f"OR={ai_row['odds_ratio']:.3f}, p={ai_row['p_value']:.4f})으로 나타났다. "
    f"반면 `ai_use_sum × it_org_any` 상호작용항은 {direction(h2_row['coefficient'])} 방향(coef={h2_row['coefficient']:.3f}{p_stars(h2_row['p_value'])}, "
    f"OR={h2_row['odds_ratio']:.3f}, p={h2_row['p_value']:.4f})으로 나타나, 기존 OLS 주분석에서 관찰된 AI 활용 범위와 IT 조직 보유의 결합 패턴이 이항화 DV에서도 유지됨을 보여준다. "
    f"`ai_use_sum × dmi` 항은 {direction(h3_row['coefficient'])} 방향(coef={h3_row['coefficient']:.3f}{p_stars(h3_row['p_value'])}, "
    f"OR={h3_row['odds_ratio']:.3f}, p={h3_row['p_value']:.4f})으로, 기존 OLS와 마찬가지로 headline으로 삼기에는 약하거나 불안정한 패턴으로 해석된다. "
    "따라서 본 강건성 분석은 인과효과가 아니라 통제변수를 포함한 조건부 관련성의 관점에서, H2의 핵심 패턴이 1~3 대 4~5의 이항 기준에서도 대체로 유지되는지 확인하는 보조 증거로 제시할 수 있다."
)

display(Markdown(paper_paragraph))

if SAVE_OUTPUTS:
    paragraph_path = OUTPUT_DIR / "binary_logit_effect_proc_improve_high_interpretation_4_6.md"
    paragraph_path.write_text(paper_paragraph + "\n", encoding="utf-8")
    print("본문용 문단 저장:", paragraph_path.relative_to(BASE_DIR))


## 10. 실행 요약

In [ ]:
print("생성 노트북: code/robustness_binary_logit_effect_proc_high.ipynb")
print("사용 데이터 파일:", DATA_PATH.relative_to(BASE_DIR))
print("데이터 shape:", LOADED_SHAPE)
print("H2 headline formula:", H2_HEADLINE_FORMULA)
print("Integrated binary logit formula:", INTEGRATED_BINARY_LOGIT_FORMULA)
print("결과 저장 폴더:", TABLE_DIR.relative_to(BASE_DIR))
